## 🧬 NGS Mini‑Pipeline Training

## 🧭 Mapping reads to a reference genome

So far, we have explored what sequencing data looks like. We now have FASTQ files containing short DNA reads.

The next step is to determine **where these reads come from in the genome**.

This process is called **alignment** or **mapping**.

---

### 🧠 What does "mapping" mean?

Each read is a short DNA sequence. On its own, it has no context.

Mapping answers the question:

👉 *Where in the reference genome does this sequence best fit?*

To do this, we compare each read against a **reference genome** (e.g. human genome) and find the best matching location.

---

### 🔬 Analogy

Think of it like this:

- The **reference genome** is a large book  
- Each **read** is a small sentence fragment  
- Mapping is the process of finding **where that sentence belongs in the book**

---

### ⚙️ Tools used

We will use:

- **BWA** → to perform the alignment  
---



The following command will map our sample2_R1.fastq  and sample2_R2.fastq to a small reference file I created for this trainig resources. The file created is a SAM file

In [3]:
!bwa mem ../mini_reference.fa ../sample2_R1.fastq ../sample2_R2.fastq > aligned.sam

/bin/bash: bwa: command not found


As we did with the FASTQ file, we can view the aligned.sam with the command cat

In [5]:
cat  aligned.sam

This is a real mapping command we have in our genotyping pipeline

In [ ]:
bwa mem -Y -L 50 -K 100000000 -v 3 $bwarefgenomepath FASTQ_R1.fastq FASTQ_R2.fastq | \
java -Djava.io.tmpdir=$PWD -Xmx8G -jar picard.jar \
    MergeBamAlignment \
    REFERENCE_SEQUENCE=$bwarefgenomepath \
    UNMAPPED_BAM=input.bam \
    ALIGNED_BAM=/dev/stdin \
    OUTPUT=sample_FINAL_mapped.bam \
    CREATE_INDEX=true \
    VALIDATION_STRINGENCY=STRICT \
    ATTRIBUTES_TO_RETAIN=X0 \
    SORT_ORDER=coordinate \
    IS_BISULFITE_SEQUENCE=false \
    ALIGNED_READS_ONLY=false \
    CLIP_ADAPTERS=false \
    MAX_RECORDS_IN_RAM=2000000 \
    ADD_MATE_CIGAR=false \
    MAX_INSERTIONS_OR_DELETIONS=-1 \
    PRIMARY_ALIGNMENT_STRATEGY=BestMapq \
    ALIGNER_PROPER_PAIR_FLAGS=false \
    ADD_PG_TAG_TO_READS=false \
    UNMAP_CONTAMINANT_READS=false \
    ATTRIBUTES_TO_REMOVE=NM \
    ATTRIBUTES_TO_REMOVE=MD

Above is how is in the pipeline but I have created a code with comments in between to allow you to understand what is going on

In [ ]:
# -------------------------------
# Step 1: Align reads with BWA
# -------------------------------

bwa mem \
    -Y \                         # Use soft clipping for supplementary alignments (better compatibility with downstream tools)
    -L 50 \                      # Clipping penalty (controls how ends of reads are handled)
    -K 100000000 \               # Process reads in large chunks (performance optimisation)
    -v 3 \                       # Verbosity level (logging detail)
    $bwarefgenomepath \          # Reference genome (must be indexed)
    FASTQ_R1.fastq \             # Forward reads
    FASTQ_R2.fastq \             # Reverse reads

# Pipe output directly to next tool (no intermediate file)
| \

# -------------------------------
# Step 2: Process alignment with Picard
# -------------------------------

java \
    -Djava.io.tmpdir=$PWD \      # Use current directory for temporary files
    -Xmx8G \                     # Allocate 8 GB of RAM
    -jar picard.jar \            # Run Picard tools

    MergeBamAlignment \          # Tool to merge alignment with metadata

    REFERENCE_SEQUENCE=$bwarefgenomepath \   # Reference genome
    UNMAPPED_BAM=input.bam \                 # Original unmapped BAM (contains metadata)
    ALIGNED_BAM=/dev/stdin \                 # Take aligned reads from BWA (via pipe)

    OUTPUT=sample_FINAL_mapped.bam \         # Final output BAM file
    CREATE_INDEX=true \                     # Create BAM index (.bai)

    VALIDATION_STRINGENCY=STRICT \          # Enforce strict format validation
    SORT_ORDER=coordinate \                 # Sort reads by genomic position

    IS_BISULFITE_SEQUENCE=false \           # Not bisulfite sequencing
    ALIGNED_READS_ONLY=false \              # Keep both mapped and unmapped reads
    CLIP_ADAPTERS=false \                  # Do not trim adapters at this stage

    MAX_RECORDS_IN_RAM=2000000 \            # Memory control for processing
    ADD_MATE_CIGAR=false \                 # Do not add mate CIGAR tags

    MAX_INSERTIONS_OR_DELETIONS=-1 \        # No limit on indels
    PRIMARY_ALIGNMENT_STRATEGY=BestMapq \   # Choose best alignment by mapping quality

    ALIGNER_PROPER_PAIR_FLAGS=false \       # Do not rely on aligner pairing flags
    ADD_PG_TAG_TO_READS=false \             # Do not add program group tags

    UNMAP_CONTAMINANT_READS=false \         # Do not unmap suspected contaminants

    ATTRIBUTES_TO_RETAIN=X0 \               # Keep specific tag (aligner-specific)
    ATTRIBUTES_TO_REMOVE=NM \               # Remove edit distance tag
    ATTRIBUTES_TO_REMOVE=MD                # Remove mismatch string tag

## About the Author

<p align="center">
  <img src="https://raw.githubusercontent.com/Manuel-DominguezCBG/ngs-teaching-binder/main/assets/Manolo.jpg" width="120" style="border-radius: 6px;">
</p>

**Manuel — Principal Bioinformatician**

Manuel is a Principal Bioinformatician specialising in the development and maintenance of clinical next‑generation sequencing (NGS) pipelines for rare disease and oncology diagnostics. His work includes the provisioning and configuration of Oracle Linux servers for clinical bioinformatics, infrastructure automation with Ansible and Git, and the management of NGS and critical document archives to ensure data integrity, accessibility, and security.

Manuel leads bioinformatics training for NHS STP Bioinformatics and rotational trainees. His work focuses on developing clear, accessible educational resources and provide hands‑on experience with real computational workflows. This notebook forms part of a personal initiative to support structured, practice‑based learning in genomics.

Feedback, adaptations, and contributions to this teaching resource are welcome.

**LinkedIn:** [Manuel J. Domínguez](https://www.linkedin.com/in/manuel-j-dom%C3%ADnguez-aa97981b2/)  
**GitHub Repository:** [ngs‑teaching‑binder](https://github.com/Manuel-DominguezCBG/ngs-teaching-binder)

**Huge thanks to the Bioinformatics team and all my WGLS colleagues for helping me make these learning resources better.**